# AETHER_CRYSTAL Option Pricing — Round 4 Manual Challenge

Verifies fair values for the 12 products with both Black-Scholes closed-form (vanillas) and Monte Carlo (everything), then compares against market quotes.

**Key spec to keep straight:**
- 1 "week" = 5 *trading* days
- "2 weeks" → T = 10/252, 60 monitoring steps total but expiry at step 40
- "3 weeks" → T = 15/252, 60 monitoring steps to expiry
- 4 steps per trading day; discrete monitoring (matters for KO)
- σ = 2.51 (annualized log-vol), r = q = 0, S₀ = 50

In [1]:
# imports & constants
import numpy as np
from scipy.stats import norm

S0    = 50.0
SIGMA = 2.51
R     = 0.0      # zero risk-neutral drift per spec
TPY   = 252      # trading days per year
SPD   = 4        # steps per trading day
SPY   = TPY * SPD

# Time to expiry in YEARS (NOT calendar days)
T_2WK = 10 / TPY        # ≈ 0.03968
T_3WK = 15 / TPY        # ≈ 0.05952

# Number of MC steps along each path
N_2WK = 2 * 5 * SPD     # 40
N_3WK = 3 * 5 * SPD     # 60

print(f"σ√T(3wk) = {SIGMA*np.sqrt(T_3WK):.4f}")
print(f"σ√T(2wk) = {SIGMA*np.sqrt(T_2WK):.4f}")

σ√T(3wk) = 0.6124
σ√T(2wk) = 0.5000


In [2]:
# Black-Scholes closed forms (r = q = 0)
def bs_d1(S, K, T, sigma):
    return (np.log(S/K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))

def bs_call(S, K, T, sigma):
    d1 = bs_d1(S, K, T, sigma); d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*norm.cdf(d2)

def bs_put(S, K, T, sigma):
    d1 = bs_d1(S, K, T, sigma); d2 = d1 - sigma*np.sqrt(T)
    return K*norm.cdf(-d2) - S*norm.cdf(-d1)

def bs_binary_put(S, K, T, sigma, payout=1.0):
    """Cash-or-nothing put paying `payout` if S_T < K. Under r=0: pay·N(-d2)."""
    d1 = bs_d1(S, K, T, sigma); d2 = d1 - sigma*np.sqrt(T)
    return payout * norm.cdf(-d2)

In [3]:
# Closed-form vanilla & binary table
print("\n--- BSM closed-form (continuous monitoring) ---")
print(f"{'product':<14s}{'fair':>10s}")
for K in [35, 40, 45, 50, 60]:
    print(f"P{K}_3wk        {bs_put (S0, K, T_3WK, SIGMA):>10.4f}")
    print(f"C{K}_3wk        {bs_call(S0, K, T_3WK, SIGMA):>10.4f}")
print(f"P50_2wk        {bs_put (S0, 50, T_2WK, SIGMA):>10.4f}")
print(f"C50_2wk        {bs_call(S0, 50, T_2WK, SIGMA):>10.4f}")
print(f"BinPut40_3wk   {bs_binary_put(S0, 40, T_3WK, SIGMA, payout=10.0):>10.4f}")

# Chooser identity (under r=0, K=ATM):
#   choose at t1 the side that is ITM == choose the side with greater value (parity)
#   t0 fair = C(S0, K, T_expire) + P(S0, K, T_choose)
chooser_cf = bs_call(S0, 50, T_3WK, SIGMA) + bs_put(S0, 50, T_2WK, SIGMA)
print(f"Chooser_50     {chooser_cf:>10.4f}    (= C(50,15td) + P(50,10td))")

# KO put has no clean closed form under discrete monitoring → MC only


--- BSM closed-form (continuous monitoring) ---
product             fair
P35_3wk            4.3361
C35_3wk           19.3361
P40_3wk            6.5095
C40_3wk           16.5095
P45_3wk            9.0889
C45_3wk           14.0889
P50_3wk           12.0269
C50_3wk           12.0269
P60_3wk           18.7918
C60_3wk            8.7918
P50_2wk            9.8707
C50_2wk            9.8707
BinPut40_3wk       4.7679
Chooser_50        21.8977    (= C(50,15td) + P(50,10td))


In [4]:
# Monte Carlo simulation
np.random.seed(2026)
N_PATHS = 500_000
dt = 1.0 / SPY

# log-increments for the FULL 60-step path (covers both 2wk and 3wk)
Z = np.random.randn(N_PATHS, N_3WK).astype(np.float32)
log_inc = (-0.5 * SIGMA**2) * dt + SIGMA * np.sqrt(dt) * Z
log_S = np.log(S0) + np.cumsum(log_inc, axis=1)
S_path = np.exp(log_S)

S_t1 = S_path[:, N_2WK - 1]   # value at end of step 40 (2 weeks)
S_T  = S_path[:, N_3WK - 1]   # value at end of step 60 (3 weeks)
S_min = S_path.min(axis=1)    # min over all 60 monitored points

In [5]:
# MC payoffs and fair values
PAYOFFS = {
    'P35_3wk':  np.maximum(35 - S_T, 0),
    'P40_3wk':  np.maximum(40 - S_T, 0),
    'P45_3wk':  np.maximum(45 - S_T, 0),
    'P50_3wk':  np.maximum(50 - S_T, 0),
    'C50_3wk':  np.maximum(S_T - 50, 0),
    'C60_3wk':  np.maximum(S_T - 60, 0),
    'P50_2wk':  np.maximum(50 - S_t1, 0),
    'C50_2wk':  np.maximum(S_t1 - 50, 0),
    'Chooser':  np.where(S_t1 >= 50,
                         np.maximum(S_T - 50, 0),
                         np.maximum(50 - S_T, 0)),
    'BinPut40': np.where(S_T < 40, 10.0, 0.0),
    'KO_Put45': np.where(S_min >= 35, np.maximum(45 - S_T, 0), 0.0),
}

print("\n--- Monte Carlo ({:,} paths) ---".format(N_PATHS))
print(f"{'product':<14s}{'fair':>10s}{'std':>10s}{'MC_SE':>10s}{'SE@100':>10s}")
for name, p in PAYOFFS.items():
    fair = p.mean()
    sd   = p.std()
    mc_se = sd / np.sqrt(N_PATHS)
    se100 = sd / np.sqrt(100)
    print(f"{name:<14s}{fair:>10.4f}{sd:>10.4f}{mc_se:>10.4f}{se100:>10.4f}")


--- Monte Carlo (500,000 paths) ---
product             fair       std     MC_SE    SE@100
P35_3wk           4.3422    6.9368    0.0098    0.6937
P40_3wk           6.5170    8.8565    0.0125    0.8857
P45_3wk           9.0988   10.7649    0.0152    1.0765
P50_3wk          12.0392   12.6150    0.0178    1.2615
C50_3wk          12.0216   26.2141    0.0371    2.6214
C60_3wk           8.7908   23.4109    0.0331    2.3411
P50_2wk           9.8999   10.9841    0.0155    1.0984
C50_2wk           9.8536   19.8599    0.0281    1.9860
Chooser          21.9198   24.7342    0.0350    2.4734
BinPut40          4.7721    4.9948    0.0071    0.4995
KO_Put45          0.2037    1.0813    0.0015    0.1081


In [6]:
# Cross-check: BSM vs MC for vanillas should agree to ~3 decimals
print("\n--- Sanity: BSM vs MC ---")
checks = [
    ('P35_3wk', bs_put(S0,35,T_3WK,SIGMA)),
    ('P40_3wk', bs_put(S0,40,T_3WK,SIGMA)),
    ('P45_3wk', bs_put(S0,45,T_3WK,SIGMA)),
    ('P50_3wk', bs_put(S0,50,T_3WK,SIGMA)),
    ('C50_3wk', bs_call(S0,50,T_3WK,SIGMA)),
    ('C60_3wk', bs_call(S0,60,T_3WK,SIGMA)),
    ('P50_2wk', bs_put(S0,50,T_2WK,SIGMA)),
    ('C50_2wk', bs_call(S0,50,T_2WK,SIGMA)),
    ('BinPut40', bs_binary_put(S0,40,T_3WK,SIGMA,10.0)),
    ('Chooser',  chooser_cf),
]
print(f"{'product':<12s}{'MC':>10s}{'BSM':>10s}{'diff':>8s}")
for name, cf in checks:
    mc = PAYOFFS[name].mean()
    print(f"{name:<12s}{mc:>10.4f}{cf:>10.4f}{mc-cf:>+8.4f}")


--- Sanity: BSM vs MC ---
product             MC       BSM    diff
P35_3wk         4.3422    4.3361 +0.0060
P40_3wk         6.5170    6.5095 +0.0075
P45_3wk         9.0988    9.0889 +0.0099
P50_3wk        12.0392   12.0269 +0.0123
C50_3wk        12.0216   12.0269 -0.0053
C60_3wk         8.7908    8.7918 -0.0010
P50_2wk         9.8999    9.8707 +0.0292
C50_2wk         9.8536    9.8707 -0.0171
BinPut40        4.7721    4.7679 +0.0042
Chooser        21.9198   21.8977 +0.0222


In [7]:
# Market quotes & per-option edge
MARKET = {  # (bid, ask, max_size)
    'P35_3wk':  (4.33, 4.35, 50),
    'P40_3wk':  (6.50, 6.55, 50),
    'P45_3wk':  (9.05, 9.10, 50),
    'P50_3wk':  (12.00, 12.05, 50),
    'C50_3wk':  (12.00, 12.05, 50),
    'C60_3wk':  (8.80, 8.85, 50),
    'P50_2wk':  (9.70, 9.75, 50),
    'C50_2wk':  (9.70, 9.75, 50),
    'Chooser':  (22.20, 22.30, 50),
    'BinPut40': (5.00, 5.10, 50),
    'KO_Put45': (0.150, 0.175, 500),
}

print("\n--- Edges per option ---")
print(f"{'product':<12s}{'fair':>8s}{'bid':>8s}{'ask':>8s}{'sell_edge':>11s}{'buy_edge':>10s}{'best':>6s}")
for name, p in PAYOFFS.items():
    fair = p.mean()
    bid, ask, _ = MARKET[name]
    sell_edge = bid - fair    # edge if you SHORT at bid
    buy_edge  = fair - ask    # edge if you LONG at ask
    best = "SELL" if sell_edge > buy_edge else "BUY"
    if max(sell_edge, buy_edge) < 0.01: best = "skip"
    print(f"{name:<12s}{fair:>8.3f}{bid:>8.3f}{ask:>8.3f}{sell_edge:>+11.3f}{buy_edge:>+10.3f}{best:>6s}")


--- Edges per option ---
product         fair     bid     ask  sell_edge  buy_edge  best
P35_3wk        4.342   4.330   4.350     -0.012    -0.008  skip
P40_3wk        6.517   6.500   6.550     -0.017    -0.033  skip
P45_3wk        9.099   9.050   9.100     -0.049    -0.001  skip
P50_3wk       12.039  12.000  12.050     -0.039    -0.011  skip
C50_3wk       12.022  12.000  12.050     -0.022    -0.028  skip
C60_3wk        8.791   8.800   8.850     +0.009    -0.059  skip
P50_2wk        9.900   9.700   9.750     -0.200    +0.150   BUY
C50_2wk        9.854   9.700   9.750     -0.154    +0.104   BUY
Chooser       21.920  22.200  22.300     +0.280    -0.380  SELL
BinPut40       4.772   5.000   5.100     +0.228    -0.328  SELL
KO_Put45       0.204   0.150   0.175     -0.054    +0.029   BUY


In [8]:
# Portfolio EV + risk under contest's 100-sim mark
CONTRACT_SIZE = 3000

def evaluate_portfolio(positions, n_contests=10000, seed=1):
    """positions = {product: signed quantity}. Negative = short.
       Returns expected PnL, std, P(positive), 5/50/95 percentiles.
    """
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, N_PATHS, size=(n_contests, 100))
    pnl = np.zeros(n_contests)
    ev = 0.0
    for prod, qty in positions.items():
        if qty == 0: continue
        bid, ask, _ = MARKET[prod]
        entry = ask if qty > 0 else bid
        contest_marks = PAYOFFS[prod][idx].mean(axis=1)   # noisy fair across 100 sims
        true_fair = PAYOFFS[prod].mean()
        pnl += qty * (contest_marks - entry) * CONTRACT_SIZE
        ev  += qty * (true_fair    - entry) * CONTRACT_SIZE
    return {
        'EV': ev,
        'mean': pnl.mean(),
        'std':  pnl.std(),
        'p_positive': (pnl > 0).mean(),
        'p5':  np.percentile(pnl, 5),
        'p50': np.percentile(pnl, 50),
        'p95': np.percentile(pnl, 95),
        'pnl': pnl,
    }

# Recommended portfolio
RECOMMENDED = {
    'P50_2wk':  +50,
    'C50_2wk':  +50,
    'Chooser':  -50,
    'BinPut40': -50,
    'KO_Put45': +500,
}

res = evaluate_portfolio(RECOMMENDED)
print("\n--- Recommended portfolio ---")
for k, v in RECOMMENDED.items():
    print(f"  {k:<10s} {v:+5d}")
print(f"True EV   : {res['EV']:+12,.0f}")
print(f"Mean PnL  : {res['mean']:+12,.0f}")
print(f"Std PnL   : {res['std']:12,.0f}")
print(f"P(+ PnL)  : {res['p_positive']*100:5.1f}%")
print(f"5/50/95   : {res['p5']:+,.0f} / {res['p50']:+,.0f} / {res['p95']:+,.0f}")


--- Recommended portfolio ---
  P50_2wk      +50
  C50_2wk      +50
  Chooser      -50
  BinPut40     -50
  KO_Put45    +500
True EV   :     +157,253
Mean PnL  :     +161,350
Std PnL   :      345,946
P(+ PnL)  :  68.2%
5/50/95   : -400,738 / +163,166 / +727,512
